In [3]:
# --- CLUSTERIZAÇÃO: PREPARAÇÃO ---
# Por que normalizar? O K-Means mede distância entre pontos. Se uma coluna vai de 0 a 100 e outra vai de 0 a 100000, a segunda vai dominar o modelo injustamente. O StandardScaler coloca tudo na mesma escala.

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.model_utils import (
    preparar_clusterizacao,
    avaliar_k_elbow_silhouette,
    treinar_kmeans,
    rotular_clusters,
    plotar_zonas_vulnerabilidade,
 )

PROCESSED = PROJECT_ROOT / 'data' / 'processed'
df_final = pd.read_parquet(PROCESSED / 'base_diamante_es_vfinal.parquet')
print(f"✅ Dados carregados: {df_final.shape}")

# No notebook 03, logo após carregar o parquet
print("Colunas com 'saude' ou 'morbidade' ou 'Taxa':")
print([c for c in df_final.columns if any(x in c.lower() for x in ['saude', 'morbi', 'taxa', 'doenca', 'cid'])])

print("\nColunas completamente nulas:")
print(df_final.columns[df_final.isna().all()].tolist())

# Features que representam risco social combinado
features_cluster = [
    'vazio_sanitario',           # déficit de infraestrutura
    'Taxa_Morbidade_100k_Hab',   # impacto na saúde
    'indice_tratamento_esgoto',  # qualidade do esgoto
    'indice_perda_distribuicao_agua',  # ineficiência da rede
    'RISCO_SOCIAL_FINAL'         # índice composto
 ]

ano_ref = int(df_final['ano'].max())
df_check = df_final[df_final['ano'] == ano_ref][features_cluster]
print("Ano ref:", ano_ref)
print("Linhas:", len(df_check))
print("Linhas completas:", len(df_check.dropna()))
print("Colunas 100% nulas:", df_check.columns[df_check.isna().all()].tolist())

df_cluster, X_scaled, scaler, ano_ref = preparar_clusterizacao(df_final, features_cluster)

print("✅ Dataset pronto para clusterização.")
print(f"   Ano de referência: {ano_ref}")
print(f"   Municípios com dados completos: {len(df_cluster)}")

✅ Dados carregados: (1267, 69)
Colunas com 'saude' ou 'morbidade' ou 'Taxa':
['tem_dado_saude_agua', 'tem_dado_saude_esgoto', 'tem_dado_saude', 'Taxa_Morbidade_100k_Hab', 'sem_dados_saude', 'vetor_dominante_doenca']

Colunas completamente nulas:
['Taxa_Morbidade_100k_Hab']
Ano ref: 2022
Linhas: 78
Linhas completas: 0
Colunas 100% nulas: ['Taxa_Morbidade_100k_Hab']


ValueError: Found array with 0 sample(s) (shape=(0, 5)) while a minimum of 1 is required by StandardScaler.

In [2]:
print(df_final.groupby('ano')['Taxa_Morbidade_100k_Hab'].count())

ano
2006    0
2007    0
2008    0
2009    0
2010    0
2011    0
2012    0
2013    0
2014    0
2015    0
2016    0
2017    0
2018    0
2019    0
2020    0
2021    0
2022    0
Name: Taxa_Morbidade_100k_Hab, dtype: int64


In [ ]:
# --- MÉTODO DO COTOVELO ---
# Como funciona: Testamos de 2 a 9 clusters e medimos o "erro interno" (inércia). O ponto onde a curva dobra como um cotovelo é o número ideal — a partir daí, adicionar mais clusters traz pouco ganho.

inercias, silhouettes, _, _ = avaliar_k_elbow_silhouette(X_scaled)
plt.show()

print("\n💡 Escolha o k onde a inércia 'dobra' E a silhueta é alta.")
print("   Geralmente entre 3 e 5 clusters para dados municipais.")

In [ ]:
# --- TREINAR K-MEANS ---
# Ajuste K_FINAL com base no gráfico da célula anterior
K_FINAL = 4

df_cluster, perfil, km_final = treinar_kmeans(
    df_cluster, X_scaled, features_cluster, K_FINAL
 )

# Perfil de cada cluster (médias das features originais)
print("📊 PERFIL DOS CLUSTERS (médias por grupo):\n")
print(perfil.to_string())

In [ ]:
# --- NOMEAR CLUSTERS (ajuste os rótulos conforme o perfil gerado acima) ---
# Ordena clusters por RISCO_SOCIAL_FINAL crescente e atribui rótulos
df_cluster, rotulos = rotular_clusters(df_cluster, perfil, K_FINAL)

# Distribuição
print("\n📊 DISTRIBUIÇÃO DOS MUNICÍPIOS POR ZONA:\n")
print(df_cluster['zona_vulnerabilidade'].value_counts().to_string())

# Scatter: Vazio Sanitário vs Morbidade, colorido por zona
plotar_zonas_vulnerabilidade(df_cluster, ano_ref)
plt.show()

# Propaga os rótulos para o df_final
df_final = df_final.merge(
    df_cluster[['id_municipio', 'cluster', 'zona_vulnerabilidade']],
    on='id_municipio', how='left'
 )
print("\n✅ Zonas propagadas para df_final.")

In [ ]:
# --- EXPLICABILIDADE DO MODELO ---
from sklearn.metrics import silhouette_score

sil = silhouette_score(X_scaled, km_final.labels_)
print(f"Coeficiente de Silhueta (k={K_FINAL}): {sil:.4f}")
print("  → Acima de 0.5: clusters bem separados | Entre 0.3-0.5: estrutura razoável\n")

centroides_orig = scaler.inverse_transform(km_final.cluster_centers_)
df_centroides = pd.DataFrame(centroides_orig, columns=features_cluster)
df_centroides.index = [rotulos[i] for i in range(K_FINAL)]
print("📊 CENTROIDES POR ZONA (valores originais):\n")
print(df_centroides.round(2).to_string())

In [ ]:
# --- MAPA COROPLÉTICO DAS ZONAS DE VULNERABILIDADE ---
# Requer: pip install geopandas folium

import geopandas as gpd
import folium
from folium.features import GeoJsonTooltip


gdf = gpd.read_file(PROJECT_ROOT / 'data' / 'raw' / 'ES_Municipios_2022.zip')

# Padroniza chave de join
gdf['id_municipio'] = gdf['CD_MUN'].astype(str)
df_cluster['id_municipio'] = df_cluster['id_municipio'].astype(str)

# Join com os clusters
gdf_mapa = gdf.merge(
    df_cluster[['id_municipio', 'zona_vulnerabilidade', 'RISCO_SOCIAL_FINAL']],
    on='id_municipio',
    how='left'
)
gdf_mapa = gdf_mapa.to_crs(epsg=4326)

# Mapa interativo com Folium
mapa = folium.Map(location=[-19.9, -40.3], zoom_start=7, tiles='CartoDB positron')

cores_zona = {
    'Zona Verde — Baixo Risco':      '#2ecc71',
    'Zona Amarela — Risco Moderado': '#f1c40f',
    'Zona Laranja — Risco Elevado':  '#e67e22',
    'Zona Vermelha — Risco Crítico': '#e74c3c',
}

def estilo(feature):
    zona = feature['properties'].get('zona_vulnerabilidade', None)
    return {
        'fillColor': cores_zona.get(zona, '#bdc3c7'),
        'color': 'white',
        'weight': 0.5,
        'fillOpacity': 0.8
    }

folium.GeoJson(
    gdf_mapa,
    style_function=estilo,
    tooltip=GeoJsonTooltip(
        fields=['NM_MUN', 'zona_vulnerabilidade', 'RISCO_SOCIAL_FINAL'],
        aliases=['Município:', 'Zona:', 'Risco Social:'],
        localize=True
    )
).add_to(mapa)

mapa.save('mapa_vulnerabilidade_es.html')
print("✅ Mapa salvo como 'mapa_vulnerabilidade_es.html'")
mapa  # Exibe no Colab

In [ ]:
# --- EXPORTAR PARQUET FINAL ENRIQUECIDO ---
PROCESSED = PROJECT_ROOT / 'data' / 'processed'
df_final.to_parquet(PROCESSED / 'base_final_com_zonas.parquet', index=False)
print(f"✅ Exportado com {df_final.shape[0]} linhas e {df_final.shape[1]} colunas.")
print(f"   Colunas novas: zona_vulnerabilidade, cluster, RISCO_SOCIAL_FINAL")